**Exploration des données**

In [12]:
import numpy as np
import pandas as pd

In [13]:
df = pd.read_csv("dataset.csv")

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5304 entries, 0 to 5303
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   diseases   5304 non-null   object
 1   symptoms   5304 non-null   object
 2   diagnosis  5304 non-null   object
dtypes: object(3)
memory usage: 124.4+ KB


**Aperçu des données**

In [15]:
df.head()

,diseases,symptoms,diagnosis
0,Psoriasis,"The skin around my mouth, nose, and eyes is re...","Based on your description, it seems like you m..."
1,Psoriasis,The rash on my skin is worse in the winter mon...,It sounds like you are experiencing a worsenin...
2,Psoriasis,The skin on my palms and soles is thickened an...,"Based on your description, it seems like you m..."
3,Psoriasis,I have noticed a sudden peeling of skin at dif...,"I'm not a dermatologist, but based on your des..."
4,Psoriasis,I am starting to have rashes on my skin. The r...,"Based on your description, it is difficult to ..."


**Description du dataset**

In [16]:
df.describe(include='all')

,diseases,symptoms,diagnosis
count,5304,5304,5304
unique,2163,4821,5304
top,Malaria,"Swelling, pain, dry mouth, bad taste","Based on your description, it seems like you m..."
freq,101,6,1


In [17]:
# Renommons les colonnes pour l'entraînement 
df = df.rename(columns={'diseases': 'maladies', 
                          'symptoms': 'symptomes', 
                          'diagnosis': 'diagnostics'})
df.head()

,maladies,symptomes,diagnostics
0,Psoriasis,"The skin around my mouth, nose, and eyes is re...","Based on your description, it seems like you m..."
1,Psoriasis,The rash on my skin is worse in the winter mon...,It sounds like you are experiencing a worsenin...
2,Psoriasis,The skin on my palms and soles is thickened an...,"Based on your description, it seems like you m..."
3,Psoriasis,I have noticed a sudden peeling of skin at dif...,"I'm not a dermatologist, but based on your des..."
4,Psoriasis,I am starting to have rashes on my skin. The r...,"Based on your description, it is difficult to ..."


**Nettoyage des données**

In [18]:

df = df.dropna(how="any").reset_index(drop=True)  

print("Colonnes:", df.columns.tolist())
print("Nombre de lignes:", len(df))


Colonnes: ['maladies', 'symptomes', 'diagnostics']
Nombre de lignes: 5304


In [19]:
#filtrage des labels peu fréquents pour garder les maladies d'au moins 3 apparitions
LABEL_COL="maladies"
counts = df[LABEL_COL].value_counts()
valid_labels = counts[counts >= 3].index.tolist()
print(f"Labels initiaux: {len(counts)}.Labels conservés : {len(valid_labels)}")

df = df[df[LABEL_COL].isin(valid_labels)].reset_index(drop=True)
print("Taille après filtrage:", df.shape)


Labels initiaux: 2163.Labels conservés : 119
Taille après filtrage: (2946, 3)


**Les synonymes des termes médicales en anglais**

In [20]:
medical_synonyms = {
    "fever": ["pyrexia", "high temperature", "raised temperature"],
    "headache": ["cephalgia", "migraine pain", "cranial pain"],
    "cough": ["tussis", "dry cough", "wet cough"],
    "fatigue": ["tiredness", "exhaustion", "weariness"],
    "chills": ["shivering", "rigors", "cold sensation"],
    "nausea": ["queasiness", "sickness", "upset stomach"],
    "vomiting": ["emesis", "throwing up", "retching"],
    "diarrhea": ["loose stools", "frequent stools", "bowel upset"],
    "abdominal pain": ["stomach ache", "belly pain", "gastric pain"],
    "shortness of breath": ["dyspnea", "breathlessness", "difficulty breathing"],
    "chest pain": ["thoracic pain", "pressure chest", "angina-like pain"],
    "rash": ["skin eruption", "dermal rash", "skin irritation"],
    "itching": ["pruritus", "skin itch", "irritation"],
    "dizziness": ["vertigo", "lightheadedness", "imbalance"],
    "sore throat": ["pharyngitis", "throat pain", "irritated throat"],
    "runny nose": ["rhinorrhea", "nasal discharge", "dripping nose"],
    "stuffy nose": ["nasal congestion", "blocked nose", "clogged nose"],
    "joint pain": ["arthralgia", "joint stiffness", "joint ache"],
    "muscle pain": ["myalgia", "soreness", "muscle ache"],
    "weight loss": ["unintentional weight loss", "slimming", "cachexia"],
    "weight gain": ["overweight", "obesity onset", "increased body mass"],
    "high blood pressure": ["hypertension", "elevated BP"],
    "low blood pressure": ["hypotension", "low BP"],
    "rapid heart rate": ["tachycardia", "fast heartbeat"],
    "slow heart rate": ["bradycardia", "slow pulse"],
    "blurred vision": ["vision fog", "visual blurring"],
    "eye pain": ["ocular pain", "orbital pain"],
    "ear pain": ["otalgia", "ear ache"],
    "hearing loss": ["reduced hearing", "auditory loss"],
    "back pain": ["lumbar pain", "spinal pain"],
    "neck pain": ["cervical pain", "neck stiffness"],
    "anxiety": ["nervousness", "stress", "panic"],
    "depression": ["low mood", "sadness", "depressive feeling"],
    "confusion": ["disorientation", "mental fog", "cognitive difficulty"],
    "swelling": ["edema", "inflammation", "fluid retention"],
    "infection": ["contamination", "pathogen invasion"],
    "inflammation": ["swelling", "irritation"],
    "flu": ["influenza", "viral infection"],
    "cold": ["common cold", "upper respiratory infection"],
    "covid": ["covid-19", "coronavirus infection"],
    "diabetes": ["hyperglycemia", "sugar imbalance"],
    "asthma": ["airway inflammation", "bronchospasm"],
    "bronchitis": ["airway infection", "bronchial inflammation"],
    "pneumonia": ["lung infection", "pulmonary inflammation"],
    "tb": ["tuberculosis", "mycobacterial infection"],
    "malaria": ["plasmodium infection", "parasitic fever"],
    "migraine": ["head pain", "pulsating headache"],
    "ulcer": ["gastric ulcer", "stomach sore"],
    "allergy": ["hypersensitivity", "allergic reaction"],
    "stroke": ["cerebrovascular accident", "brain attack"],
    "heart attack": ["myocardial infarction", "cardiac arrest"],
    "kidney pain": ["renal pain", "flank pain"],
    "liver pain": ["hepatic pain"],
    "joint swelling": ["arthritic swelling"],
    "diabetic pain": ["neuropathic pain", "nerve pain"],
    "burning sensation": ["burning pain", "neuropathic burning"],
    "cramps": ["spasms", "muscle tightening"],
    "fainting": ["syncope", "loss of consciousness"],
    "bleeding": ["hemorrhage", "blood loss"],
    "constipation": ["difficulty bowel movement"],
    "palpitations": ["irregular heartbeat", "fluttering heart"],
    "tremor": ["shaking", "involuntary movement"],
    "weakness": ["loss of strength", "reduced strength"],
    "paralysis": ["loss of movement", "motor deficit"],
    "sweating": ["perspiration", "excessive sweating"],
    "infection": ["bacterial infection", "viral infection"],
    "inflammation": ["irritation", "swelling"],
    "stroke": ["brain ischemia", "cerebral infarction"],
    "fracture": ["broken bone"],
    "pain": ["ache", "discomfort", "soreness"]
}


**La fonction pour ajouter les synonymes médicaux**

In [21]:
def ajouter_synonymes(clean_text):
    words = clean_text.split()
    expanded = []
    for w in words:
        expanded.append(w)
        if w in medical_synonyms:
            expanded.extend(medical_synonyms[w])
    return " ".join(expanded)


In [22]:
#!pip install spacy
#Installation du modèle anglais
#!python -m spacy download en_core_web_sm


**Nettoyage avec Spacy, nlp, nltk et ajout des synonymes**

In [23]:
import spacy   
from nltk.corpus import stopwords
import nltk
import re

nlp = spacy.load("en_core_web_sm")
nltk.download('stopwords', quiet=True)
eng_stop = set(stopwords.words('english'))

# variable manquante
use_spacy = True

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    #Lemmatisation
    if use_spacy:
        doc = nlp(text)
        tokens = [tok.lemma_ for tok in doc 
                  if tok.is_alpha and not tok.is_stop and len(tok.lemma_) > 1]
    else:
        tokens = [t for t in text.split() 
                  if t not in eng_stop and len(t) > 1]

    text = " ".join(tokens)

    # Ajout des synonymes 
    text = ajouter_synonymes(text)

    return text


OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

**Application du nettoyage et affichage des symptomes initiaux et des symptomes nettoyés**

In [13]:
# Application du nettoyage 
df["symptomes_clean"] = df["symptomes"].apply(clean_text)

df[["symptomes", "symptomes_clean"]].head()

,symptomes,symptomes_clean
0,"The skin around my mouth, nose, and eyes is re...",skin mouth nose eye red inflame itchy uncomfor...
1,The rash on my skin is worse in the winter mon...,rash skin eruption dermal rash skin irritation...
2,The skin on my palms and soles is thickened an...,skin palm sol thicken deep crack crack painful...
3,I have noticed a sudden peeling of skin at dif...,notice sudden peel skin different part body ma...
4,I am starting to have rashes on my skin. The r...,start rash skin eruption dermal rash skin irri...


**Chargement du modèle et encodage des symptômes**

In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Chargement du modèle 
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# Encodage du dataset
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

symptom_embeddings = model.encode(
    df['symptomes_clean'].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/93 [00:00<?, ?it/s]

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
from deep_translator import GoogleTranslator

**Fonction de traduction de l'entrée de l'utilisateur français-anglais et vice versa**

In [16]:
def translate_fr_en(txt):
    if not txt or not isinstance(txt, str):
        return ""
    try:
        return GoogleTranslator(source='fr', target='en').translate(txt)
    except:
        return txt

def translate_en_fr(txt):
    if not txt or not isinstance(txt, str):
        return ""
    try:
        return GoogleTranslator(source='en', target='fr').translate(txt)
    except:
        return txt


In [17]:
# Correction orthographique 

def correct_spelling(text):
    
    return text


**Fonction de prédiction**

In [18]:
def predict_disease(user_text, top_k=3):
    #Traduction du texte utilisateur FR à EN
    text_en = translate_fr_en(user_text)

    #Correction orthographique du texte d'utlisateur
    text_corrected = correct_spelling(text_en)

    #Nettoyage 
    text_clean = clean_text(text_corrected)

    # texte vide après nettoyage, on vérifie si la longueur du texte nettoyé est inférieure à 2 pour afficher ce message
    if len(text_clean.split()) < 2:
        return [{
            "maladie": None,
            "message": (
                "Les informations fournies sur vos symptômes semblent insuffisantes pour permettre une analyse fiable. "
                "Afin de mieux comprendre votre situation, pourriez-vous préciser la localisation, l’intensité ainsi que la durée de vos symptômes ?"
            ),
            "similarité": 0,
            "seuil_dynamique": 0
        }]
  

    # Calcul du score d’overlap 
    user_words = set(text_clean.split())

    df["overlap_score"] = df["symptomes_clean"].apply(
        lambda x: len(user_words.intersection(str(x).split()))
    )

    overlap_max = df["overlap_score"].max()

    #Embeddings utilisateur
    user_vector = model.encode([text_clean])

    # Application de la Similarité cosine 
    similarities = cosine_similarity(user_vector, symptom_embeddings)[0]

    best_idx = similarities.argsort()[-top_k:][::-1]
    best_similarity = similarities[best_idx[0]]

    # Seuil dynamique 
    dynamic_threshold = max(np.percentile(similarities, 95), 0.30)
    low_conf_threshold = dynamic_threshold * 0.50
    medium_conf_threshold = dynamic_threshold * 0.70

    # Cas de très faible confiance
    if best_similarity < low_conf_threshold or overlap_max == 0:
        return [{
            "maladie": None,
            "message": (
                "Je ne parviens pas à interpréter correctement vos symptômes. "
                "Pouvez-vous donner plus de détails par exemple localisation, intensité, durée?"
            ),
            "similarité": float(best_similarity),
            "seuil_dynamique": float(dynamic_threshold)
        }]

    # Cas de confiance moyenne 
    if low_conf_threshold <= best_similarity < medium_conf_threshold:
        return [{
            "maladie": None,
            "message": (
                "Vos symptômes correspondent partiellement à certains cas connus, "
                "mais la précision est insuffisante. "
                "Pouvez-vous préciser la durée, l’intensité ou d’autres symptômes associés ?"
            ),
            "similarité": float(best_similarity),
            "seuil_dynamique": float(dynamic_threshold)
        }]

    # Cas de forte confiance pour prédire maintenant la maladie
    predictions = []
    for idx in best_idx:
        pred = {
            "maladie": df.iloc[idx]["maladies"],
            "symptome_dataset": df.iloc[idx]["symptomes"],
            "diagnostic": df.iloc[idx]["diagnostics"],
            "similarité": float(similarities[idx]),
            "seuil_dynamique": float(dynamic_threshold),
            "overlap": int(df.iloc[idx]["overlap_score"])
        }
        predictions.append(pred)

    # Traduction des diagnostics de l'utilisateur en français
    for p in predictions:
        p["maladie"]   = translate_en_fr(p["maladie"])
        p["diagnostic"] = translate_en_fr(p["diagnostic"])
        p["symptome_dataset"] = translate_en_fr(p["symptome_dataset"])


    return predictions


**Test de la foncton de prédiction**

In [19]:
test = predict_disease("Je sens des des douleurs au niveau de mes os")
test


[{'maladie': 'Leucémie',
  'symptome_dataset': "Si je souffre d'un «\xa0gonflement\xa0» et de «\xa0douleurs osseuses et articulaires\xa0», de quels tests dois-je discuter\xa0?",
  'diagnostic': "Si vous ressentez un gonflement et des douleurs osseuses et articulaires, vous pouvez discuter de plusieurs tests avec votre médecin pour vous aider à déterminer la cause de vos symptômes. Ces tests peuvent inclure\xa0: 1. Radiographies\xa0: les radiographies peuvent fournir des images détaillées de vos os et de vos articulations, aidant ainsi à identifier toute fracture, luxation ou anomalie.  2. Imagerie par résonance magnétique (IRM) : Une IRM utilise de puissants aimants et des ondes radio pour créer des images détaillées de vos os, articulations et tissus mous. Cela peut aider à détecter des blessures, une inflammation ou des problèmes structurels.  3. Analyses sanguines\xa0: les analyses de sang peuvent être utiles pour identifier certaines affections pouvant provoquer un gonflement et de

**Modèle machine learning:Support Vector Machine, SVM**

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
#Nettoyage unique 
df["symptomes_clean"] = df["symptomes"].apply(clean_text)

#Séparation 
X_train, X_test, y_train, y_test = train_test_split(
    df["symptomes_clean"],
    df["maladies"],
    test_size=0.2,
    random_state=42
)

emb_train = model.encode(X_train.tolist())
emb_test  = model.encode(X_test.tolist())

# --- SVM final 
modele_svm = SVC(
    kernel='linear',
    probability=True,
    C=5,
    class_weight='balanced'
)

modele_svm.fit(emb_train, y_train)

preds_svm = modele_svm.predict(emb_test)

acc_svm = accuracy_score(y_test, preds_svm)
print("Accuracy SVM :", acc_svm)


NameError: name 'df' is not defined

**Sauve

In [21]:
import joblib
import os
import shutil
import tempfile
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

SAVE_DIR = "models"
os.makedirs(SAVE_DIR, exist_ok=True)

#  Sauvegarde SVM
joblib.dump(modele_svm, os.path.join(SAVE_DIR, "svm_classifier.pkl"))

#  Sauvegarde SBERT en .pkl
temp_dir = tempfile.mkdtemp()
model.save(temp_dir)

# Puis on compresse le dossier dans un fichier .pkl
joblib.dump(temp_dir, os.path.join(SAVE_DIR, "sbert_encoder.pkl"))

#  Sauvegarde embeddings
np.save(os.path.join(SAVE_DIR, "symptom_embeddings.npy"), symptom_embeddings)

# Sauvegarde df
# ------------------------------
df.to_csv(os.path.join(SAVE_DIR, "df_index.csv"), index=False, encoding="utf-8")

print("Modèles sauvegardés avec succès.")


Modèles sauvegardés avec succès.
